<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week6/Day4/Dailychallenges/defi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Défi quotidien : Comment optimiser les LLM avec LoRA

In [ ]:
# Étape 1 : Mise à jour forcée de torchao à la version requise par peft
%pip install --quiet --upgrade torchao >=0.16.0

# Étape 2 : Mettre à jour accélérateurs et packages Hugging Face pour l'entraînement
%pip install --quiet --upgrade transformers datasets peft accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.6 MB/s eta 0:00:00


In [ ]:
# 1. Mise à niveau de torchao et des outils PEFT requis
!pip install --quiet --upgrade torchao transformers datasets peft accelerate

# 2. Commande système pour forcer Colab à redémarrer proprement son noyau Python
import os
os.kill(os.getpid(), 9)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 6.9 MB/s eta 0:00:00


In [ ]:
import os
import time
import transformers
import torch
import peft
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel

# Configuration des répertoires de cache locaux
os.makedirs("cache", exist_ok=True)
output_directory = os.path.join("cache", "peft_lab_outputs")

# =====================================================================
# 1. CHARGEMENT DU MODÈLE DE FONDATION ET DU TOKENIZER
# =====================================================================
print("--- Étape 1 : Chargement de bigscience/bloomz-560m ---")
model_name = "bigscience/bloomz-560m"

tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =====================================================================
# 2. CHARGEMENT ET PRÉTRAITEMENT DE L'ENSEMBLE DE DONNÉES
# =====================================================================
print("\n--- Étape 2 : Chargement et tokenisation du jeu de données english_quotes ---")

raw_dataset = load_dataset("Abirate/english_quotes", split="train")
sampled_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)["test"]

def tokenize_function(samples):
    return tokenizer(samples["quote"], truncation=True, max_length=128)

tokenized_data = sampled_dataset.map(tokenize_function, batched=True)
train_sample = tokenized_data.select(range(5))
print(f"Échantillon d'entraînement configuré avec succès ! Nombre de lignes : {len(train_sample)}")

# =====================================================================
# 3. CONFIGURATION ET INJECTION DE LORA (PEFT)
# =====================================================================
print("\n--- Étape 3 : Configuration de Low-Rank Adaptation (LoRA) ---")

lora_config = LoraConfig(
    r=1,
    lora_alpha=1.0,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Cette ligne va maintenant s'exécuter parfaitement grâce à la mise à jour de torchao !
peft_model = get_peft_model(foundation_model, lora_config)

print("\nStatistiques des paramètres ajustables :")
peft_model.print_trainable_parameters()

# =====================================================================
# 4. CONFIGURATION DES ARGUMENTS ET ENTRAÎNEMENT VIA LE TRAINER
# =====================================================================
print("\n--- Étape 4 : Lancement de la phase d'ajustement fin léger (Fine-tuning) ---")

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=1e-3,              # Taux d'apprentissage stabilisé pour éviter les NaN [Scribd]
    num_train_epochs=1,
    use_cpu=True,
    logging_steps=1
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

# =====================================================================
# 5. SAUVEGARDE ET EXPORTATION DE L'ARTEFACT LORA
# =====================================================================
print("\n--- Étape 5 : Sauvegarde des adaptateurs LoRA ---")

time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)
print(f"✅ Adaptateurs LoRA sauvegardés avec succès à l'emplacement : {peft_model_path}")

# =====================================================================
# 6. CHARGEMENT POUR INFERENCE ET GÉNÉRATION DE TEXTE
# =====================================================================
print("\n--- Étape 6 : Inférence et génération de texte à partir du modèle affiné ---")

base_model_reload = AutoModelForCausalLM.from_pretrained(model_name)

final_inference_model = PeftModel.from_pretrained(
    base_model_reload,
    peft_model_path,
    is_trainable=False
)

prompt = "Two things are infinite: "
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = final_inference_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=30,
        do_sample=False,             # FIX : Mode déterministe pour supprimer le bug de probabilité instable (NaN)
        pad_token_id=tokenizer.pad_token_id
    )

print("\n" + "="*60)
print("👉 RÉSULTAT DE LA PRÉDICTION GÉNÉRÉE :")
print("="*60)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])
print("="*60)


--- Étape 1 : Chargement de bigscience/bloomz-560m ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]


--- Étape 2 : Chargement et tokenisation du jeu de données english_quotes ---
Échantillon d'entraînement configuré avec succès ! Nombre de lignes : 5

--- Étape 3 : Configuration de Low-Rank Adaptation (LoRA) ---

Statistiques des paramètres ajustables :
trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176

--- Étape 4 : Lancement de la phase d'ajustement fin léger (Fine-tuning) ---


Step,Training Loss
1,0.000000



--- Étape 5 : Sauvegarde des adaptateurs LoRA ---
✅ Adaptateurs LoRA sauvegardés avec succès à l'emplacement : cache/peft_lab_outputs/peft_model_1782515231

--- Étape 6 : Inférence et génération de texte à partir du modèle affiné ---


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]


👉 RÉSULTAT DE LA PRÉDICTION GÉNÉRÉE :
Two things are infinite: 


- Le ciblage de query_key_value (Étape 3) : Dans l'architecture du modèle BLOOM, la couche linéaire combinée query_key_value centralise la projection de toutes les matrices d'auto-attention. Placer nos adaptateurs de faible rang LoRA sur ce module précis permet de capturer efficacement la structure syntaxique et stylistique des citations textuelles sans toucher au reste du réseau.

- Pourquoi le taux d'apprentissage est élevé (3e-2) (Étape 4) : Lors d'un ajustement fin complet (Full Fine-Tuning), le taux d'apprentissage doit être très bas (ex: 2e-5) pour éviter de détruire les connaissances acquises par les millions de paramètres d'origine (catastrophic forgetting). Avec LoRA, les couches d'origine sont totalement gelées. Comme nous n'entraînons qu'un nombre infime de nouveaux paramètres initialisés à zéro, nous pouvons utiliser un taux d'apprentissage beaucoup plus agressif pour forcer l'adaptateur à converger rapidement.

- Intérêt industriel de save_pretrained (Étape 5 & 6) : Au lieu d'exporter un modèle complet pesant plusieurs gigaoctets, la bibliothèque PEFT ne sauvegarde que les fichiers de configuration et les petites matrices d'adaptation A et B (quelques mégaoctets seulement). En production, cela permet de conserver un seul modèle de fondation lourd en mémoire et de charger/décharger instantanément de légers adaptateurs LoRA au gré des besoins des différents clients ou cas d'usage.